In [432]:
import os
from pathlib import Path

import autoroot  # noqa: F401
import polars as pl
from sklearn.model_selection import train_test_split

df = pl.read_csv("hf://datasets/dllllb/rosbank-churn/train.csv.gz").filter(currency=810)

In [433]:
df_processed = (
    df.with_columns(
        datetime=pl.col("TRDATETIME").str.to_datetime("%d%b%y:%T"),
        MCC=pl.col("MCC").rank("dense").cast(pl.Int32),
        amount=pl.col("amount").abs().log1p(),
        target="target_flag",
    )
    .filter(
        (pl.col("datetime").max() - pl.col("datetime").min()).over("cl_id")
        > pl.duration(weeks=1),
        pl.len().over("cl_id") > 32,
    )
    .with_columns(
        time=(pl.col("datetime") - pl.col("datetime").min()).over("cl_id")
        / (pl.col("datetime").max() - pl.col("datetime").min()).over("cl_id").median()
    )
    .drop(
        "PERIOD",
        "datetime",
        "channel_type",
        "currency",
        "trx_category",
        "TRDATETIME",
        "target_flag",
        "target_sum",
    )
    .group_by("cl_id")
    .agg(
        pl.exclude("target").sort_by("time"),
        pl.col("target").first(),
    )
)


In [434]:
df_trainval, df_test = train_test_split(
    df_processed, test_size=1024, random_state=42, stratify=df_processed["target"]
)
df_train, df_val = train_test_split(
    df_trainval, test_size=256, stratify=df_trainval["target"]
)

df_train = df_train.with_columns(split=pl.lit("train"))
df_val = df_val.with_columns(split=pl.lit("val"))
df_test = df_test.with_columns(split=pl.lit("test"))

df = pl.concat([df_train, df_val, df_test])

In [435]:
df.write_parquet(Path(os.environ["DATA_DIR"]) / "preprocessed/churn.parquet")